# 30 — Assemble Tier 3 DataBooks

*Reads the COSMoS graph plus the Variable-Identity, Test-Identity and
Domain-Metadata reference files; applies the template catalogue in
`sdtm-narrative/reference/templates/` to emit one markdown DataBook
per case pair (Glucose, 6MWT, X-Ray). HTML views are rendered from the
canonical markdown.*

**Build order per the plan:** Glucose (Templates 01 + 04), 6MWT
(Template 02), X-Ray (Templates 03 + 04).

**Data integrity caveat.** Three facts from the reconnaissance step,
material to what these DataBooks can truthfully say:

- Glucose BC (C105585) has 8 DSSs in domain LB. Full data for
  Template 01.
- 6MWT BC (C115789) has `bc_type = full_no_ds` — the BC exists but
  no DSS rows are published. The 6MWT DataBook therefore renders the
  BC scaffolding only; the item-level rendering that Template 02
  describes is structural proposal, not current graph content.
- X-Ray BC (C38101) has 2 DSSs — both in domain PR. The cross-
  Domain_Class framing in `XRay_COSMoS_Story.html` (MK + PR) is the
  story's prospective argument, not current graph content. The
  DataBook renders the two PR DSSs and flags the MK gap.

These are flagged in-line in each DataBook as explicit notes, not
papered over. Template 02's registry-need band and Template 03's
registry-need band *are the published content* for 6MWT and X-Ray
respectively — that is consistent with their reference stories,
which are as much gap arguments as graph renderings.

## Setup — paths, imports, loads

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

REPO = Path("/Users/kerstinforsberg/repos/GitHub/cdisc-for-ai")

GRAPH       = REPO / "cosmos-graph/interim/COSMoS_Graph.xlsx"
CT          = REPO / "cosmos-graph/interim/COSMoS_Graph_CT.xlsx"
VAR_IDENT   = REPO / "sdtm-narrative/reference/SDTM_Variable_Identity.xlsx"
TEST_IDENT  = REPO / "sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx"
INSTR_IDENT = REPO / "sdtm-test-codes/machine_actionable/SDTM_Instrument_Identity.xlsx"
DOMAIN_META = REPO / "sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx"

OUT_DIR = REPO / "sdtm-narrative/machine_actionable/databooks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

bc         = pd.read_excel(GRAPH, sheet_name="BC")
dss        = pd.read_excel(GRAPH, sheet_name="DSS")
variables  = pd.read_excel(GRAPH, sheet_name="Variables")
codelists  = pd.read_excel(GRAPH, sheet_name="Codelists")

ct_at      = pd.read_excel(CT, sheet_name="AssignedTerms")
ct_cl      = pd.read_excel(CT, sheet_name="Codelists")
ct_terms   = pd.read_excel(CT, sheet_name="CodelistTerms")

var_ident    = pd.read_excel(VAR_IDENT, sheet_name="Variable_Identity")
test_ident   = pd.read_excel(TEST_IDENT)
instr_ident  = pd.read_excel(INSTR_IDENT, sheet_name="Instruments")
domain_meta  = pd.read_excel(DOMAIN_META, sheet_name="Domains")

print(f"BC rows       : {len(bc):>6}")
print(f"DSS rows      : {len(dss):>6}")
print(f"Variable rows : {len(variables):>6}")
print(f"Var identity  : {len(var_ident):>6}")
print(f"Test identity : {len(test_ident):>6}")
print(f"Domain meta   : {len(domain_meta):>6}")

## Two-tier natural-name substitution

Per the catalogue `00_index.md` rule: direct hit first (non-root
subset), compositional fallback second (strip two-char domain prefix,
look up `--REMAINDER` against Root subset).

In [ ]:
# Split the identity table into direct vs root halves
# Identity sheet columns are Title-Case: Variable, NCIt_Code, Natural_Name, Definition, Subset, Applicable_Domains, Notes
_direct_tbl = (var_ident[var_ident['Subset'] != 'Root']
               .drop_duplicates(subset=['Variable'], keep='first')
               .set_index('Variable'))
_root_tbl   = (var_ident[var_ident['Subset'] == 'Root']
               .drop_duplicates(subset=['Variable'], keep='first')
               .set_index('Variable'))

def var_nn(variable, with_definition=False):
    """Return the natural-English name for an SDTM variable code.

    Two-tier rule: direct identity first; if not present, strip the
    two-character domain prefix and look up --REMAINDER against the
    Root subset.  Returns (name, definition, source) when
    with_definition=True, else just name.
    """
    if variable in _direct_tbl.index:
        row = _direct_tbl.loc[variable]
        out = (row['Natural_Name'], row['Definition'], 'direct')
    elif len(variable) > 2 and variable[2:] in _root_tbl.index:
        row = _root_tbl.loc[variable[2:]]
        out = (row['Natural_Name'], row['Definition'], 'compositional')
    else:
        out = (variable, '', 'unresolved')
    return out if with_definition else out[0]

# Smoke checks
for v in ['LBTESTCD', 'LBSPEC', 'LBMETHOD', 'LBFAST', 'MKMETHOD', 'QSCAT', 'PRDECOD']:
    n, d, src = var_nn(v, with_definition=True)
    print(f"{v:10} → {src:14} {n}")

## Structural-type resolver for a DSS

Template 01 (specimen-based) vs Template 02 (instrument-based) vs
Template 03 (cross-Domain_Class) is decided by the DSS's domain
(joined to `SDTM_Domain_Metadata`) and by whether the BC binds
multiple Domain_Class values.

In [ ]:
# Domain → structural-type mapping
def structural_type(domain_code):
    row = domain_meta[domain_meta['Domain'] == domain_code]
    if not len(row):
        return 'unknown'
    r = row.iloc[0]
    if r['Observation_Class'] != 'Findings':
        return 'non-findings'
    if r['Specimen_Based'] == 'Yes':
        return 'specimen-based-findings'
    if r['Measurement'] == 'Yes':
        return 'measurement-findings'
    return 'instrument-findings'

def bc_domain_classes(bc_id):
    """Return distinct structural types bound under a BC."""
    domains = dss[dss['bc_id'] == bc_id]['domain'].dropna().unique().tolist()
    return sorted({structural_type(d) for d in domains})

for bc_id in ['C105585', 'C115789', 'C38101']:
    bname = bc[bc['bc_id']==bc_id]['bc_short_name'].iloc[0] if (bc['bc_id']==bc_id).any() else '—'
    print(f"{bc_id}  {bname!r}  → {bc_domain_classes(bc_id)}")

## DSS attribute extraction from Variables rows

In the real graph schema, specimen / method / result-scale / unit
attributes are not DSS-sheet columns; they are **Variables** rows
with pinned (`assigned_term_value`) or constrained (`value_list` /
`codelist_submission_value`) values. This helper extracts a
Template-01-shaped attribute dict from the Variables rows for a DSS.

In [ ]:
def dss_attributes(ds_id):
    """Extract specimen/method/result/unit attributes from Variables rows."""
    v = variables[variables['ds_id'] == ds_id]
    d = dss[dss['ds_id'] == ds_id].iloc[0]
    dom = d['domain']

    def pick(var_name):
        row = v[v['variable_name'] == var_name]
        return row.iloc[0] if len(row) else None

    attrs = {'ds_id': ds_id, 'domain': dom}
    for (short, suffix) in [('specimen','SPEC'), ('method','METHOD'),
                            ('category','CAT'), ('sub_category','SCAT'),
                            ('orig_unit','ORRESU'), ('std_unit','STRESU'),
                            ('loinc','LOINC'), ('fast','FAST'),
                            ('position','POS'), ('location','LOC'),
                            ('decod','DECOD')]:
        row = pick(dom + suffix)
        if row is None:
            attrs[short] = None
            continue
        # Pinned value takes precedence; else value_list / codelist describes the open slot
        if pd.notna(row.get('assigned_term_value')):
            attrs[short] = {'pinned': row['assigned_term_value'],
                            'concept_id': row.get('assigned_term_concept_id'),
                            'codelist': row.get('codelist_submission_value')}
        elif pd.notna(row.get('value_list')):
            attrs[short] = {'open_list': row['value_list'],
                            'codelist': row.get('codelist_submission_value')}
        elif pd.notna(row.get('codelist_submission_value')):
            attrs[short] = {'open_codelist': row['codelist_submission_value']}
        else:
            attrs[short] = {'open_free': True}
    return attrs

# Smoke
from pprint import pprint
pprint(dss_attributes('GLUCPL'))

## Case-specialisation stub registry

Per the catalogue §04 and design-record §5, the case registry is not
yet a graph sheet. For Tier 3 validation we encode the two reference
cases — Glucose StudyIntent and X-Ray PatientBurden — as literal
rows extracted from the corresponding HTML stories. Provenance is the
story file path; values are copied verbatim from the story content,
not generated.

When §5 resolves, this stub moves into
`sdtm-narrative/reference/case_specialisations.xlsx` (or the chosen
promotion target) and this cell becomes a file load.

In [ ]:
# Literal facts from docs/Glucose_StudyIntent_Story.html
# (provenance: that file, authored 2026-Q1).
CASE_SPECS = [
    {
        'case_spec_id'    : 'ID001',
        'case_label'      : 'Fasting Plasma Glucose (FPG)',
        'parent_dss'      : 'GLUCPL',
        'case_type'       : 'recording-child',
        'composes_with'   : None,
        'pinning'         : [('LBFAST', 'Y', 'pinned on a slot the DSS already reserves')],
        'ext_ref'         : None,
        'rationale'       : 'Fasting sample carries the clinical diagnostic intent',
        'source_file'     : 'docs/Glucose_StudyIntent_Story.html',
    },
    {
        'case_spec_id'    : 'ID002',
        'case_label'      : 'OGTT 2-Hour Plasma Glucose',
        'parent_dss'      : 'GLUCPL',
        'case_type'       : 'recording-child',
        'composes_with'   : 'OGTT 75g oral challenge (SoA-scheduling parent, study-design object)',
        'pinning'         : [('LBFAST', 'N', 'post-challenge sample')],
        'ext_ref'         : None,
        'rationale'       : 'Post-challenge sample under OGTT 75g protocol',
        'source_file'     : 'docs/Glucose_StudyIntent_Story.html',
    },
    {
        'case_spec_id'    : 'ID003',
        'case_label'      : 'OGTT Fasting (Pre-Challenge) Plasma Glucose',
        'parent_dss'      : 'GLUCPL',
        'case_type'       : 'recording-child',
        'composes_with'   : 'OGTT 75g oral challenge (same SoA-scheduling parent as ID002)',
        'pinning'         : [('LBFAST', 'Y', 'pre-challenge baseline')],
        'ext_ref'         : None,
        'rationale'       : 'Pre-challenge baseline under OGTT 75g protocol',
        'source_file'     : 'docs/Glucose_StudyIntent_Story.html',
    },
    # Literal facts from docs/XRay_PatientBurden_Story.html
    {
        'case_spec_id'    : 'ID101',
        'case_label'      : 'Standing PA Chest X-Ray',
        'parent_dss'      : 'XRAYCHEST',
        'case_type'       : 'recording-child + SoA-activity',
        'composes_with'   : None,
        'pinning'         : [],  # inherits parent pins only; no additional slot available
        'ext_ref'         : None,
        'rationale'       : 'Standard diagnostic standing PA; inherits XRAYCHEST pins (PRDECOD=X-RAY, PRLOC=CHEST).',
        'source_file'     : 'docs/XRay_PatientBurden_Story.html',
    },
    {
        'case_spec_id'    : 'ID102',
        'case_label'      : 'Supine Portable Chest X-Ray',
        'parent_dss'      : 'XRAYCHEST',
        'case_type'       : 'recording-child + SoA-activity',
        'composes_with'   : None,
        'pinning'         : [],
        'ext_ref'         : None,
        'rationale'       : 'Portable supine; inherits XRAYCHEST pins; no additional slot available.',
        'source_file'     : 'docs/XRay_PatientBurden_Story.html',
    },
    {
        'case_spec_id'    : 'ID103',
        'case_label'      : 'Semi-Recumbent Chest X-Ray (Post-Op Follow-Up)',
        'parent_dss'      : 'XRAYCHEST',
        'case_type'       : 'recording-child + SoA-activity',
        'composes_with'   : 'Post-operative assessment event (SoA-scheduling parent, study-design object)',
        'pinning'         : [],
        'ext_ref'         : None,
        'rationale'       : 'Post-operative follow-up positioning; inherits XRAYCHEST pins.',
        'source_file'     : 'docs/XRay_PatientBurden_Story.html',
    },
]
case_specs_for = {}
for c in CASE_SPECS:
    case_specs_for.setdefault(c['parent_dss'], []).append(c)

print(f"Case-spec registry: {len(CASE_SPECS)} rows covering parents {sorted(case_specs_for)}")

## Band builders

Markdown fragments follow the catalogue under
`sdtm-narrative/reference/templates/`. Each builder returns a list
of lines; DataBook assembly concatenates them.

In [ ]:
def fmt_slot(slot):
    """Render a DSS attribute slot (pinned / open / None) as markdown."""
    if slot is None:
        return '—'
    if 'pinned' in slot:
        return f"**{slot['pinned']}** *(pinned)*"
    if 'open_list' in slot:
        return f"open, constrained to `{slot['open_list']}`"
    if 'open_codelist' in slot:
        return f"open, codelist `{slot['open_codelist']}`"
    if 'open_free' in slot:
        return 'open'
    return '—'


def band1_specimen(ds_row, bc_row, attrs):
    bname = bc_row['bc_short_name']
    dname = ds_row['ds_short_name']
    did   = ds_row['ds_id']
    dom   = ds_row['domain']
    testcd_row = variables[(variables['ds_id']==did) & (variables['variable_name']==dom+'TESTCD')]
    testcd_val = testcd_row['assigned_term_value'].iloc[0] if len(testcd_row) else '—'
    since = ds_row.get('sdtmig_start_version', '')
    out = [f"""The **{dname}** specialisation (`{did}`) realises the **{bname}** biomedical concept as a {dom}-domain row template. It pins `{dom}TESTCD = {testcd_val}`."""]
    if pd.notna(since) and since:
        out[-1] += f" Added to SDTMIG in {since}."
    return out


def band3_pinning_table(ds_id, roles=('Topic','Qualifier','Timing')):
    v = variables[variables['ds_id']==ds_id]
    out = ["", "| Variable | Natural name | Role | Pinned value | Codelist | Value list |",
           "|---|---|---|---|---|---|"]
    for _, r in v.iterrows():
        if r['role'] not in roles:
            continue
        nn = var_nn(r['variable_name'])
        pinned = r['assigned_term_value'] if pd.notna(r['assigned_term_value']) else ''
        cl = r['codelist_submission_value'] if pd.notna(r['codelist_submission_value']) else ''
        vl = r['value_list'] if pd.notna(r['value_list']) else ''
        out.append(f"| `{r['variable_name']}` | {nn} | {r['role']} | {pinned} | {cl} | {vl} |")
    return out


def band3_attributes_table(attrs):
    out = ["", "| Attribute | Value |", "|---|---|"]
    for k in ['specimen','method','category','sub_category','orig_unit','std_unit','fast','position','location','decod','loinc']:
        if attrs.get(k) is not None:
            out.append(f"| {k.replace('_',' ').title()} | {fmt_slot(attrs[k])} |")
    return out


def band3_siblings(bc_id, this_ds_id):
    sibs = dss[(dss['bc_id']==bc_id) & (dss['ds_id']!=this_ds_id)]
    if not len(sibs):
        return []
    out = ["", "**Sibling DSSs under the same BC:**", ""]
    for _, s in sibs.iterrows():
        out.append(f"- `{s['ds_id']}` — {s['ds_short_name']} (domain {s['domain']})")
    return out


def band4_flat_stub(ds_row, bc_row):
    """Stub band-4 rendering — reads what we have; the Flat file is not built yet."""
    out = ["", "*Flattened row (band 4) — stub rendering from graph; the",
           "`COSMoS_Graph_Flat.xlsx` round-trip file is not yet produced.*", "",
           "| Field | Value |", "|---|---|"]
    for f, v in [("BC_ID", bc_row['bc_id']),
                 ("BC_short_name", bc_row['bc_short_name']),
                 ("BC_definition", bc_row['bc_definition']),
                 ("BC_categories", bc_row['bc_categories']),
                 ("BC_type", bc_row['bc_type']),
                 ("result_scales", bc_row['result_scales']),
                 ("DS_Code", ds_row['ds_id']),
                 ("DS_short_name", ds_row['ds_short_name']),
                 ("Domain", ds_row['domain']),
                 ("sdtmig_since", ds_row.get('sdtmig_start_version',''))]:
        out.append(f"| {f} | {v if pd.notna(v) else ''} |")
    return out


def band4_case(case):
    out = ["", f"#### Case specialisation — {case['case_spec_id']}: {case['case_label']}", "",
           f"Parent DSS: `{case['parent_dss']}`  ",
           f"Case type: {case['case_type']}  "]
    if case['composes_with']:
        out.append(f"Composes with: {case['composes_with']}  ")
    if case['pinning']:
        out.append("")
        out.append("**Inside-DSS pinning (overlay on parent):**")
        for var, val, note in case['pinning']:
            out.append(f"- `{var}` = **{val}** — {note}")
    else:
        out.append("")
        out.append("*No additional inside-DSS pinning — case inherits parent pins only.*")
    if case['rationale']:
        out.append("")
        out.append(f"Rationale: {case['rationale']}")
    out.append("")
    out.append(f"*Source: `{case['source_file']}`*")
    return out


def band5_specimen_registry_gap(bc_row, ds_count_siblings):
    bname = bc_row['bc_short_name']
    return ["", f"> ### Registry gap — specimen-test qualification", ">",
            f"> The COSMoS graph models **{bname}** at the BC level and its",
            f"> {ds_count_siblings} specialisations at the DSS level, but does not",
            "> carry a first-class registry of *which test–specimen–method",
            "> combinations are clinically meaningful*. The sibling DSSs are",
            "> the practical evidence such combinations exist; the graph",
            "> represents them only as separate DSS rows, not as a qualified",
            "> proposition.", ">",
            "> A registry of shape *(BC, TESTCD, specimen, method, result_scale,",
            "> qualified_by)* would make specimen–test qualification machine-",
            "> actionable."]


def band5_instrument_registry_gap(bc_row):
    bname = bc_row['bc_short_name']
    return ["", f"> ### Registry gap — instrument composition", ">",
            f"> The **{bname}** BC carries the instrument identity at BC level",
            "> but publishes no DSS rows (`bc_type = full_no_ds`). Item-level",
            "> DSS coverage and NCIt composition ancestry would make the",
            "> instrument addressable as a first-class artefact — its items,",
            "> its published version, its evidence of validation."]


def band5_cross_domain_registry_gap(bc_row):
    bname = bc_row['bc_short_name']
    return ["", f"> ### Registry gap — cross-domain composition", ">",
            f"> The **{bname}** BC binds DSSs in the PR domain (`XRAY`,",
            "> `XRAYCHEST`). The MK-domain framing argued for in",
            "> `docs/XRay_COSMoS_Story.html` (the finding read from the image)",
            "> is prospective — it is not present in the current graph. A",
            "> cross-domain-class registry would name which Domain_Class",
            "> bindings are expected for which clinical events."]


def band5_case_registry_gap():
    return ["", "> ### Registry gap — case specialisations", ">",
            "> The cases above are encoded inline in this notebook from the",
            "> reference HTML stories. A registry of shape",
            "> *(case_spec_id, parent_dss, case_type, composes_with, rationale,",
            "> ext_ref)* would lift these from story prose into machine-",
            "> actionable first-class content."]

## DataBook 1 — Glucose (Templates 01 + 04)

In [ ]:
def build_glucose_databook():
    bc_id = 'C105585'
    bc_row = bc[bc['bc_id']==bc_id].iloc[0]
    dss_rows = dss[dss['bc_id']==bc_id].sort_values('ds_id')

    out = [f"# Glucose — COSMoS DataBook", "",
           f"*BC `{bc_id}` — {bc_row['bc_short_name']}.*",
           f"*Generated from `COSMoS_Graph.xlsx` via the template catalogue under `sdtm-narrative/reference/templates/`.*", "",
           "---", "", "## BC opener", "",
           f"**{bc_row['bc_short_name']}** ({bc_id}) is a *{bc_row['bc_type']}* biomedical concept in the {bc_row['bc_categories']} category. It realises {len(dss_rows)} dataset specialisations, all in domain `{dss_rows['domain'].unique()[0] if len(dss_rows['domain'].unique())==1 else ', '.join(dss_rows['domain'].unique())}` — a {structural_type(dss_rows['domain'].iloc[0])} pattern.",
           "",
           f"*Definition.* {bc_row['bc_definition']}",
           "",
           f"*Result scales available.* {bc_row['result_scales']}",
           "", "---", ""]

    for _, ds_row in dss_rows.iterrows():
        ds_id = ds_row['ds_id']
        attrs = dss_attributes(ds_id)
        out += ["", f"## DSS `{ds_id}` — {ds_row['ds_short_name']}", ""]
        out += band1_specimen(ds_row, bc_row, attrs)
        out += ["", "### Variables (Template 01 band 3a)"]
        out += band3_pinning_table(ds_id)
        out += ["", "### Measurement-spec attributes (band 3a)"]
        out += band3_attributes_table(attrs)
        # band 3c — case-spec stub cross-link
        cases_here = case_specs_for.get(ds_id, [])
        if cases_here:
            out += ["", "### Case specialisations refining this DSS (Template 04)"]
            for c in cases_here:
                out += band4_case(c)
        # band 4 — flat stub
        out += ["", "### Flattened row (band 4 stub)"]
        out += band4_flat_stub(ds_row, bc_row)

    out += ["", "---", "", "## BC-scope sibling summary"]
    out += band3_siblings(bc_id, this_ds_id=None)
    # list once at BC scope
    sibs = dss[dss['bc_id']==bc_id]
    out += band5_specimen_registry_gap(bc_row, len(sibs))
    out += band5_case_registry_gap()

    return "\n".join(out) + "\n"

glucose_md = build_glucose_databook()
print(glucose_md[:1200])

## DataBook 2 — 6MWT (Template 02, BC-scaffolding only)

In [ ]:
def build_6mwt_databook():
    bc_id = 'C115789'
    bc_row = bc[bc['bc_id']==bc_id].iloc[0]
    dss_rows = dss[dss['bc_id']==bc_id]

    out = ["# 6MWT — COSMoS DataBook", "",
           f"*BC `{bc_id}` — {bc_row['bc_short_name']}.*",
           f"*Generated from `COSMoS_Graph.xlsx` via the template catalogue.*", "",
           "---", "", "## BC opener", "",
           f"**{bc_row['bc_short_name']}** ({bc_id}) is registered in COSMoS with `bc_type = {bc_row['bc_type']}`. It binds {len(dss_rows)} DSSs in the current graph.",
           "",
           f"*Definition.* {bc_row['bc_definition']}",
           "",
           f"*Result scales.* {bc_row['result_scales']}",
           "", "---", "",
           "## Data-integrity note", "",
           "The `bc_type = full_no_ds` flag means this BC has no DSS rows published. The template-02 per-item rendering (DSS-grain paragraphs for DISTWLK1M and other 6MWT items) cannot be produced from the current graph. What follows is BC-level scaffolding plus the registry-need band that Template 02 publishes for instrument BCs.",
           ""]

    # NCIt ancestry from SDTM_Instrument_Identity, if available
    instr_hit = instr_ident[instr_ident['Instrument_NCIt_Code'].astype(str).str.contains('C115789', na=False)] if 'Instrument_NCIt_Code' in instr_ident.columns else pd.DataFrame()
    out += ["## Instrument composition ancestry", ""]
    if len(instr_hit):
        out += ["*From `SDTM_Instrument_Identity.xlsx`:*", "", "| Field | Value |", "|---|---|"]
        for c in instr_hit.columns:
            out.append(f"| {c} | {instr_hit.iloc[0][c]} |")
    else:
        out += ["*No matching instrument-identity row located for C115789 in the current `SDTM_Instrument_Identity.xlsx`. The 6MWT container-codelist anchor discussed in `docs/6MWT_NCIt_Story.html` is a reference-layer projection that this notebook does not currently join to.*"]

    out += band5_instrument_registry_gap(bc_row)
    return "\n".join(out) + "\n"

sixmwt_md = build_6mwt_databook()
print(sixmwt_md[:1200])

## DataBook 3 — X-Ray (Templates 03 + 04)

In [ ]:
def build_xray_databook():
    bc_id = 'C38101'
    bc_row = bc[bc['bc_id']==bc_id].iloc[0]
    dss_rows = dss[dss['bc_id']==bc_id].sort_values('ds_id')

    out = ["# Chest X-Ray — COSMoS DataBook", "",
           f"*BC `{bc_id}` — {bc_row['bc_short_name']}.*",
           f"*Generated from `COSMoS_Graph.xlsx` via the template catalogue.*", "",
           "---", "", "## BC opener", "",
           f"**{bc_row['bc_short_name']}** ({bc_id}) binds {len(dss_rows)} DSSs in the current graph, all in domain `{', '.join(dss_rows['domain'].unique())}`.",
           "", f"*Definition.* {bc_row['bc_definition']}",
           "", f"*Categories.* {bc_row['bc_categories']}",
           "", "---", "",
           "## Data-integrity note", "",
           "`docs/XRay_COSMoS_Story.html` argues a cross-Domain_Class composition where the same BC binds both a PR DSS (procedure performed) and an MK DSS (finding read from the image). The current graph carries only PR-domain DSSs for `C38101`. Template 03's cross-class band (Band 3c and Band 5 cross-domain registry-need) is therefore published here as a *gap argument*, not as a rendered binding.",
           ""]

    for _, ds_row in dss_rows.iterrows():
        ds_id = ds_row['ds_id']
        attrs = dss_attributes(ds_id)
        out += ["", f"## DSS `{ds_id}` — {ds_row['ds_short_name']}", ""]
        out += band1_specimen(ds_row, bc_row, attrs)
        out += ["", "### Variables"]
        out += band3_pinning_table(ds_id)
        out += ["", "### Attributes"]
        out += band3_attributes_table(attrs)
        cases_here = case_specs_for.get(ds_id, [])
        if cases_here:
            out += ["", "### Case specialisations refining this DSS (Template 04)"]
            for c in cases_here:
                out += band4_case(c)
        out += ["", "### Flattened row (band 4 stub)"]
        out += band4_flat_stub(ds_row, bc_row)

    out += ["", "---", "", "## Cross-Domain_Class framing"]
    out += ["", "The reference story posits an MK-domain partner DSS (morphological finding). In the current graph both DSSs are PR-domain. The cross-class proposition the BC should realise is:",
            "",
            "> *A chest X-ray, as a clinical event, is simultaneously a procedure executed (PR-domain) and a morphological observation read from the resulting image (MK-domain). Both rows belong together under this BC.*"]
    out += band5_cross_domain_registry_gap(bc_row)
    out += band5_case_registry_gap()

    return "\n".join(out) + "\n"

xray_md = build_xray_databook()
print(xray_md[:1200])

## Minimal markdown → HTML renderer

Matches the look of the reference HTML stories only loosely — bands
render as sections, tables as tables, blockquote registry-need bands
as styled callouts. Canonical source remains the markdown file; HTML
is a view.

In [ ]:
HTML_STYLE = '''
<style>
body { font: 15px/1.55 -apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
       max-width: 960px; margin: 2em auto; padding: 0 1em; color: #222; }
h1 { border-bottom: 2px solid #1F4E79; padding-bottom: 0.2em; }
h2 { color: #1F4E79; margin-top: 1.6em; border-bottom: 1px solid #ddd; padding-bottom: 0.1em; }
h3 { color: #444; margin-top: 1.4em; }
h4 { color: #666; }
code { background: #f5f5f5; padding: 0.1em 0.35em; border-radius: 3px;
       font: 13px/1.4 'SF Mono',Menlo,monospace; }
table { border-collapse: collapse; margin: 0.6em 0 1em 0; }
th, td { border: 1px solid #e0e0e0; padding: 0.35em 0.7em; text-align: left; vertical-align: top; }
th { background: #f6f8fa; }
blockquote { border-left: 4px solid #d0a84a; background: #fdf6e3; margin: 1em 0; padding: 0.6em 1em; }
blockquote h3 { margin-top: 0; color: #8a6d3b; }
em { color: #666; }
</style>
'''

def md_to_html(md_text, title):
    """Minimal markdown subset renderer — handles what the DataBooks emit."""
    lines = md_text.splitlines()
    html = []
    in_table = False
    in_quote = False
    for line in lines:
        stripped = line.rstrip()
        # Headings
        m = re.match(r'^(#{1,6})\s+(.*)$', stripped)
        if m and not in_table:
            if in_quote:
                html.append('</blockquote>'); in_quote = False
            level = len(m.group(1))
            html.append(f'<h{level}>{m.group(2)}</h{level}>')
            continue
        # Horizontal rule
        if stripped == '---':
            if in_table: html.append('</table>'); in_table=False
            if in_quote: html.append('</blockquote>'); in_quote=False
            html.append('<hr>')
            continue
        # Blockquote
        if stripped.startswith('>'):
            body = stripped.lstrip('>').lstrip()
            if not in_quote:
                html.append('<blockquote>'); in_quote = True
            m = re.match(r'^(#{1,6})\s+(.*)$', body)
            if m:
                html.append(f'<h{len(m.group(1))}>{m.group(2)}</h{len(m.group(1))}>')
            elif body:
                html.append(f'<p>{_inline(body)}</p>')
            continue
        elif in_quote and not stripped:
            html.append('</blockquote>'); in_quote = False
            continue
        # Table row
        if stripped.startswith('|') and stripped.endswith('|'):
            cells_ = [c.strip() for c in stripped.strip('|').split('|')]
            if re.match(r'^[-:\s]+$', cells_[0]):  # separator row
                continue
            if not in_table:
                html.append('<table>'); in_table = True
                html.append('<tr>' + ''.join(f'<th>{_inline(c)}</th>' for c in cells_) + '</tr>')
            else:
                html.append('<tr>' + ''.join(f'<td>{_inline(c)}</td>' for c in cells_) + '</tr>')
            continue
        elif in_table:
            html.append('</table>'); in_table = False
        # List
        if stripped.startswith('- '):
            html.append(f'<li>{_inline(stripped[2:])}</li>')
            continue
        # Paragraph
        if stripped:
            html.append(f'<p>{_inline(stripped)}</p>')
    if in_table: html.append('</table>')
    if in_quote: html.append('</blockquote>')

    body = '\n'.join(html)
    # Wrap loose <li> into <ul>
    body = re.sub(r'(?:<li>.*?</li>\s*)+', lambda m: '<ul>' + m.group(0) + '</ul>', body, flags=re.DOTALL)
    return f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><title>{title}</title>{HTML_STYLE}</head>
<body>{body}</body></html>"""

def _inline(text):
    # bold **x**
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
    # italic *x* (single asterisks not adjacent to space)
    text = re.sub(r'(?<!\*)\*([^*\n]+?)\*(?!\*)', r'<em>\1</em>', text)
    # code `x`
    text = re.sub(r'`([^`]+)`', r'<code>\1</code>', text)
    return text

## Write outputs

Canonical `.md` plus rendered `.html` per DataBook under
`sdtm-narrative/machine_actionable/databooks/`.

In [ ]:
for name, md_text in [('glucose', glucose_md),
                      ('6mwt',    sixmwt_md),
                      ('xray',    xray_md)]:
    md_path   = OUT_DIR / f"{name}.md"
    html_path = OUT_DIR / f"{name}.html"
    md_path.write_text(md_text)
    title = {'glucose':'Glucose DataBook','6mwt':'6MWT DataBook','xray':'Chest X-Ray DataBook'}[name]
    html_path.write_text(md_to_html(md_text, title))
    print(f"wrote {md_path}  ({len(md_text)} chars)")
    print(f"wrote {html_path}")

## Smoke validation

Light checks in-notebook; the full no-fabrication / round-trip
validation is task 3e (`40_validate_narrative.ipynb`).

- Every DSS rendered has a non-empty band-1 opener.
- Every variable referenced in band-3 tables has a natural-name
  resolution (direct or compositional); unresolved codes are listed.
- Case-spec rows reference parent_dss values that exist in the
  DSS sheet.

In [ ]:
import re as _re

# Unresolved variable codes referenced in the rendered DataBooks
unresolved = []
for md_text in [glucose_md, sixmwt_md, xray_md]:
    for m in _re.finditer(r'`([A-Z]{2}[A-Z0-9]{2,7})`', md_text):
        code_ = m.group(1)
        if var_nn(code_, with_definition=True)[2] == 'unresolved':
            unresolved.append(code_)
unresolved = sorted(set(unresolved))
print(f"Unresolved variable codes cited in DataBooks: {unresolved or '(none)'}")

# Case-spec parent DSS existence
for c in CASE_SPECS:
    exists = (dss['ds_id']==c['parent_dss']).any()
    flag = 'OK' if exists else 'MISSING'
    print(f"{c['case_spec_id']}  parent_dss={c['parent_dss']}  {flag}")

# Word counts
for name, m in [('glucose',glucose_md),('6mwt',sixmwt_md),('xray',xray_md)]:
    words = len(m.split())
    print(f"{name:8}  {len(m):>6} chars  {words:>5} words")

## Notes for review

- Template 01/02 catalogue entries referenced DSS-sheet columns like
  `Specimen`, `Method`, `Result_Scale` and a separate `Flat` sheet;
  the real graph schema keeps those as Variables rows (pinned value
  or open codelist). The `dss_attributes()` helper above reconciles
  the two — but the catalogue markdown should be revised in 3f to
  match the graph shape, with the Variables-row-to-attribute
  projection documented.
- Template 03 cross-Domain_Class and Template 02 instrument-item
  rendering both lack current graph data (X-Ray has PR only; 6MWT
  has no DSSs). The DataBooks publish the registry-need bands for
  those cases, which matches the reference stories' intent.
- The `COSMoS_Graph_Flat.xlsx` flat file referenced in the catalogue
  §band 4 is not produced by the current Step-2 graph build. Band 4
  here renders a graph-derived stub with BC + DSS fields. Promoting
  band 4 to true round-trip requires building (or repurposing) the
  flat file.